In [ ]:
"""
Pandas interview-practice data.

Run:
    python pandas_interview_data.py

Or import:
    from pandas_interview_data import df, df_dirty, users

Objects:
    df        - clean transactions table for exercises 1-7
    df_dirty  - same logical table, but with mixed/dirty amount values for exercise 8
    users     - user dimension table, including users with no transactions
"""

from __future__ import annotations

import numpy as np
import pandas as pd


def make_interview_data() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    # The row order is intentionally not chronological.
    transaction_rows = [
        # txn_id, user_id, txn_date, product, amount, status
        (1001, "U001", "2026-01-03 09:15", "QuickBooks", 120.00, "completed"),
        (1002, "U001", "2026-01-08 14:20", "TurboTax", 80.00, "completed"),  # <7 days
        (1003, "U001", "2026-02-17 10:00", "Mint", 0.00, "completed"),
        (1004, "U001", "2026-03-01 16:45", "QuickBooks", 75.00, "refunded"),
        (1005, "U001", "2026-03-04 08:30", "QuickBooks", 150.00, "completed"),  # <7 days
        (1006, "U002", "2026-01-15 11:00", "TurboTax", 200.00, "completed"),
        (1007, "U002", "2026-02-15 11:00", "TurboTax", 200.00, "completed"),  # tie by amount
        (1008, "U002", "2026-03-15 11:00", "QuickBooks", 95.50, "completed"),
        (1009, "U002", "2026-03-18 09:30", "Mint", 15.00, "refunded"),  # <7 days
        (1010, "U003", "2026-01-28 13:10", "Mint", 12.50, "completed"),
        (1011, "U003", "2026-02-02 13:10", "Mint", 20.00, "completed"),  # <7 days
        (1012, "U003", "2026-02-06 17:40", "TurboTax", 300.00, "completed"),  # <7 days
        (1013, "U003", "2026-04-22 09:00", "QuickBooks", 50.00, "completed"),
        (1014, "U004", "2026-02-10 10:15", "QuickBooks", 500.00, "refunded"),
        (1015, "U004", "2026-04-10 10:15", "QuickBooks", 450.00, "completed"),
        (1016, "U005", "2026-01-05 12:00", "QuickBooks", 1204.50, "completed"),
        (1017, "U005", "2026-01-25 12:00", "TurboTax", 110.00, "completed"),
        (1018, "U005", "2026-02-25 12:00", "QuickBooks", 1204.50, "completed"),
        (1019, "U005", "2026-03-25 12:00", "Mint", 5.00, "completed"),
        (1020, "U006", "2026-03-02 18:20", "TurboTax", 65.00, "completed"),
        (1021, "U006", "2026-03-02 18:25", "TurboTax", 65.00, "completed"),  # same day
        (1022, "U006", "2026-03-09 18:20", "QuickBooks", 130.00, "completed"),  # exactly 7 days
        (1023, "U007", "2026-01-31 23:55", "Mint", 9.99, "completed"),
        (1024, "U007", "2026-02-01 00:05", "Mint", 9.99, "completed"),  # crosses month, <7 days
        (1025, "U007", "2026-02-20 15:00", "TurboTax", 250.00, "refunded"),
        (1026, "U008", "2026-04-01 09:00", "QuickBooks", 90.00, "completed"),
        (1027, "U008", "2026-04-08 09:00", "QuickBooks", 90.00, "completed"),  # exactly 7 days
        (1028, "U008", "2026-04-15 09:00", "QuickBooks", 90.00, "completed"),
        (1029, "U009", "2026-02-14 10:30", "TurboTax", 180.00, "refunded"),
        (1030, "U009", "2026-02-16 10:30", "TurboTax", 190.00, "completed"),  # <7 days
        (1031, "U010", "2026-01-12 07:45", "Mint", 2.50, "completed"),
        (1032, "U010", "2026-05-12 07:45", "QuickBooks", 700.00, "completed"),
        (1033, "U011", "2026-03-30 20:00", "TurboTax", 99.00, "completed"),
        (1034, "U011", "2026-04-03 20:00", "TurboTax", 149.00, "completed"),  # <7 days
        (1035, "U011", "2026-04-05 20:00", "QuickBooks", 199.00, "refunded"),  # <7 days
        (1036, "U011", "2026-04-06 20:00", "Mint", 19.00, "completed"),  # <7 days
        (1037, "U012", "2026-05-01 10:00", "QuickBooks", 40.00, "completed"),
        (1038, "U012", "2026-05-20 10:00", "QuickBooks", 60.00, "completed"),
        (1039, "U012", "2026-06-15 10:00", "TurboTax", 85.00, "completed"),
        # Same timestamp for one user: forces a deliberate tie-breaking choice
        # when identifying the "first" transaction.
        (1040, "U013", "2026-02-01 09:00", "Mint", 11.00, "completed"),
        (1041, "U013", "2026-02-01 09:00", "TurboTax", 89.00, "completed"),
        (1042, "U013", "2026-02-08 09:00", "QuickBooks", 175.00, "completed"),
        (1043, "U014", "2026-06-01 08:00", "QuickBooks", 300.00, "refunded"),
    ]

    df = pd.DataFrame(
        transaction_rows,
        columns=["txn_id", "user_id", "txn_date", "product", "amount", "status"],
    )

    df["txn_date"] = pd.to_datetime(df["txn_date"])
    df["amount"] = df["amount"].astype("float64")

    # Shuffle deterministically so solutions must explicitly sort where required.
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)

    user_rows = [
        ("U001", "2025-12-15", "United States"),
        ("U002", "2026-01-02", "Canada"),
        ("U003", "2025-11-20", "United States"),
        ("U004", "2026-02-01", "United Kingdom"),
        ("U005", "2025-10-10", "Israel"),
        ("U006", "2026-02-20", "Canada"),
        ("U007", "2026-01-30", "France"),
        ("U008", "2026-03-15", "Germany"),
        ("U009", "2026-02-10", "United Kingdom"),
        ("U010", "2025-12-01", "Israel"),
        ("U011", "2026-03-01", "France"),
        ("U012", "2026-04-01", "Australia"),
        ("U013", "2026-01-15", "Germany"),
        ("U014", "2026-05-20", "Canada"),
        # No transactions: needed to test left joins and zero-revenue countries/users.
        ("U015", "2026-01-01", "Japan"),
        ("U016", "2026-02-14", "Brazil"),
        ("U017", "2026-06-01", "Japan"),
    ]

    users = pd.DataFrame(
        user_rows,
        columns=["user_id", "signup_date", "country"],
    )
    users["signup_date"] = pd.to_datetime(users["signup_date"])

    # Dirty version for the cleaning exercise. Only amount differs from df.
    df_dirty = df.copy()
    df_dirty["amount"] = df_dirty["amount"].astype("object")
    dirty_values = {
        1001: None,  # NaN to impute using QuickBooks median
        1006: "200.00",  # numeric string
        1012: "300",  # integer-looking string
        1016: "1,204.50",  # comma separator
        1018: "1,204.50",  # repeated formatted value
        1023: None,  # NaN to impute using Mint median
        1026: "90.00",
        1033: None,  # NaN to impute using TurboTax median
        1038: "60",
        1042: "175.00",
    }
    for txn_id, value in dirty_values.items():
        df_dirty.loc[df_dirty["txn_id"].eq(txn_id), "amount"] = value

    # Basic integrity checks; these are data checks, not exercise solutions.
    assert df["txn_id"].is_unique
    assert users["user_id"].is_unique
    assert pd.api.types.is_datetime64_any_dtype(df["txn_date"])
    assert pd.api.types.is_datetime64_any_dtype(users["signup_date"])
    assert pd.api.types.is_numeric_dtype(df["amount"])
    assert df_dirty["amount"].dtype == object
    assert set(df["product"]) == {"QuickBooks", "TurboTax", "Mint"}
    assert set(df["status"]) == {"completed", "refunded"}

    return df, df_dirty, users


df, df_dirty, users = make_interview_data()

In [3]:
display(df.sample(5))
display(df_dirty.sample(5))
display(users.sample(5))

,txn_id,user_id,txn_date,product,amount,status
11,1028,U008,2026-04-15 09:00:00,QuickBooks,90.0,completed
10,1007,U002,2026-02-15 11:00:00,TurboTax,200.0,completed
34,1036,U011,2026-04-06 20:00:00,Mint,19.0,completed
6,1005,U001,2026-03-04 08:30:00,QuickBooks,150.0,completed
30,1022,U006,2026-03-09 18:20:00,QuickBooks,130.0,completed


,txn_id,user_id,txn_date,product,amount,status
24,1042,U013,2026-02-08 09:00:00,QuickBooks,175.00,completed
22,1001,U001,2026-01-03 09:15:00,QuickBooks,None,completed
41,1029,U009,2026-02-14 10:30:00,TurboTax,180.0,refunded
27,1033,U011,2026-03-30 20:00:00,TurboTax,None,completed
40,1015,U004,2026-04-10 10:15:00,QuickBooks,450.0,completed


,user_id,signup_date,country
3,U004,2026-02-01,United Kingdom
7,U008,2026-03-15,Germany
4,U005,2025-10-10,Israel
16,U017,2026-06-01,Japan
11,U012,2026-04-01,Australia


In [4]:
# first make amounts negative for refunded transactions
df_fixed_amounts = df.copy()
df_fixed_amounts["amount"] = df_fixed_amounts["amount"] * df_fixed_amounts["status"].map({"completed": 1, "refunded": -1})
df_fixed_amounts.rename(columns={"amount": "revenue"}, inplace=True)

# now compute revenue per product, groupby product and sum amounts
revenue_per_product = df_fixed_amounts.groupby("product")["revenue"].sum().reset_index()
display(revenue_per_product)

,product,revenue
0,Mint,74.98
1,QuickBooks,3575.50
2,TurboTax,1202.00


In [10]:
# monthly revenue per product
# basically groupby month,product and sum amounts

df_fixed_amounts["month"] = df_fixed_amounts["txn_date"].dt.to_period("M")
monthly_revenue = df_fixed_amounts.groupby(["month", "product"])["revenue"].sum().reset_index()

display(monthly_revenue)

# pivot so products are cols

monthly_revenue_pivot = monthly_revenue.pivot(index="month", columns="product", values="revenue").reset_index()


display(monthly_revenue_pivot)

,month,product,revenue
0,2026-01,Mint,24.99
1,2026-01,QuickBooks,1324.50
2,2026-01,TurboTax,390.00
3,2026-02,Mint,40.99
4,2026-02,QuickBooks,879.50
5,2026-02,TurboTax,349.00
6,2026-03,Mint,-10.00
7,2026-03,QuickBooks,300.50
8,2026-03,TurboTax,229.00
9,2026-04,Mint,19.00


product,month,Mint,QuickBooks,TurboTax
0,2026-01,24.99,1324.5,390.0
1,2026-02,40.99,879.5,349.0
2,2026-03,-10.00,300.5,229.0
3,2026-04,19.00,571.0,149.0
4,2026-05,NaN,800.0,NaN
5,2026-06,NaN,-300.0,85.0


In [ ]:
import torch
from torch import nn


class LogisticModel(nn.Module):
    def __init__(self, d_input: int):
        super().__init__()

        self.d_input = d_input
        self.linear = nn.Linear(d_input, 1, bias=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input tensor of shape (batch_size, d_input)
        Returns:
            Tensor of shape (batch_size, 1) with logits
        """

        return self.linear(x)

    def probs(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input tensor of shape (batch_size, d_input)
        """

        logits = self.forward(x)
        return torch.sigmoid(logits)

    @torch.inference_mode()
    def predict(self, x: torch.Tensor, threshold: float = 0.5) -> torch.Tensor:
        """
        Args:
            x: Input tensor of shape (batch_size, d_input)
            threshold: Threshold for binary classification
        Returns:
            Tensor of shape (batch_size,) with binary predictions (0 or 1) based on the threshold
        """

        out = self.probs(x)
        return (out >= threshold).float().squeeze()

In [ ]:
from torch.utils.data import Dataset, DataLoader

# assume there is a dataset


ds_train: Dataset = None  # type: ignore
ds_test: Dataset = None  # type: ignore

dl_train = DataLoader(ds_train, batch_size=32, shuffle=True)
dl_test = DataLoader(ds_test, batch_size=32, shuffle=False)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
model = LogisticModel(d_input=10).to(device)

In [ ]:
from tqdm.auto import tqdm


epochs = 10

optim = torch.optim.Adam(model.parameters(), lr=0.001)

loss_fn = nn.BCEWithLogitsLoss()

for epoch in tqdm(range(epochs)):
    for batch in dl_train:
        x, y = batch
        x.to(device)
        y.to(device)
        
        optim.zero_grad()
        logits: torch.Tensor = model(x)
        loss = loss_fn(logits.squeeze(), y.float())
        loss.backward()
        optim.step()

In [ ]:
import math
import torch.nn.functional as F


# import attn from torch

from torch.nn import MultiheadAttention

# create attn module (single head)

class SelfAttention(nn.Module):
    def __init__(self, d_model: int, d_k: int):
        """
        Single head self-attention module.
        
        Args:
            d_model: Dimension of the input and output
            d_k: Dimension of the keys and queries
        """
        super().__init__()
        self.d_model = d_model
        self.d_k = d_k

        self.W_q = nn.Linear(d_model, d_k)
        self.W_k = nn.Linear(d_model, d_k)
        self.W_v = nn.Linear(d_model, d_k)
        
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input tensor of shape (batch_size, seq_len, d_model)
        Returns:
            Tensor of shape (batch_size, seq_len, d_model) after applying self-attention
        """
        
        Q = self.W_q(x)  # (batch_size, seq_len, d_k)
        K = self.W_k(x)  # (batch_size, seq_len, d_k)
        V = self.W_v(x)  # (batch_size, seq_len, d_k)

        # Compute attention scores (score_ij = Q_i * K_j^T / sqrt(d_k))
        
        attn_scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)  # shape (batch_size, seq_len, seq_len)
        attn_weights = F.softmax(attn_scores, dim=-1)  # shape (batch_size, seq_len, seq_len)
        
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        attn_weights = F.softmax(attn_scores, dim=-1) # shape (batch_size, seq_len, seq_len)
        
        
        out = torch.matmul(attn_weights, V)  # (batch_size, seq_len, d_k)

        return out